# GISLR-Sentences v1 — Kaggle build (self-contained)

**Generated file — do not edit.** Regenerate with
`python -m sb.recognize.sequences.kaggle_notebook` (signbridge commit
`fd53d4f`, embedded-code sha256 `1551ee0d5b3f`). This is the Kaggle-side twin of
`gislr.0.dataset.sentences.ipynb` (TODO §12.1): same code, same config, same corpus,
same seed, so it produces the same plan and the same sequences. The only
difference is where the output lands: `/kaggle/working/gislr-sentences/`, which
becomes the dataset directly with no upload from a local machine.

**How to run on Kaggle**
1. *New Notebook* → *File → Import Notebook* → this `.ipynb`.
2. *Add Input* → your dataset `bracu23101281/gislr-stratified` (it is found automatically
   under `/kaggle/input`; set `INPUT_DIR` in the setup cell if not).
3. Accelerator: **None** (CPU only; no GPU needed). Internet: not needed.
4. *Save Version* → **Save & Run All (Commit)**. The build is expected to take about
   15–30 min on Kaggle's CPUs.
5. When the version finishes: open it → *Output* tab → **New Dataset**, titled
   `GISLR-Sentences` (slug `gislr-sentences`). New datasets are **private** by default.

**Output limit.** Kaggle keeps at most ~20 GB of `/kaggle/working`. The float32 build
is estimated at ~16 GB. §3 projects the final size after the first sequences and stops
early, with instructions, if it would not fit; `STORAGE_DTYPE = "float16"` halves it.

**What it builds** (from GISLR_Stratified `test.csv` only):
- `sentence` split: 1757-sentence ASL-gloss-order corpus,
  one signer per sequence, coverage-driven so every test clip is used (~8% of slots
  re-use a clip, flagged in `reused`).
- `control` split: the same clips in random order, each exactly once.
- Synthesized null frames: 2–10 interpolated
  transition frames between signs; 10–30 rest frames
  at each end with hands absent (NaN).
- Per sequence: `landmarks (T,543,3)` in canonical row order, `frame_labels` (−1 null),
  `frame_kind` (0 sign / 1 transition / 2 rest), `segments` (start, end-exclusive), `labels`.

## Setup

In [ ]:
# ============================================================
# Tunables, config, paths
# ============================================================
import hashlib
import json
import os
import shutil
import sys
import tempfile
import types
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

INPUT_DIR = None             # None = auto-detect GISLR_Stratified under /kaggle/input
OUT_DIR = Path("/kaggle/working/gislr-sentences")
STORAGE_DTYPE = "float32"    # "float16" halves the size; segments then match the source at float16
OUTPUT_LIMIT_GB = 19.0       # Kaggle keeps ~20 GB of /kaggle/working
N_WORKERS = os.cpu_count() or 4

# configs/gislr.sentences.json, embedded verbatim at generation time
CONFIG = json.loads(r'''{
  "_comment": "GISLR-Sentences dataset build (TODO \u00a712.1) -- gislr.0.dataset.sentences.ipynb. Every tunable of the build lives here, never in a cell.",
  "dataset_version": "v1",
  "corpus_version": "v1",
  "seed": 42,
  "source_split": "test",
  "corpus": {
    "min_len": 2,
    "max_len": 8,
    "min_per_gloss": 12
  },
  "synth": {
    "gap_frames": [
      2,
      10
    ],
    "rest_frames": [
      10,
      30
    ],
    "rest_jitter": 0.002
  },
  "planner": {
    "reuse_penalty": 2.0,
    "repeat_penalty": 0.05,
    "length_bonus": 0.3,
    "pressure_weight": 0.3
  },
  "n_roundtrip_checks": 300,
  "kaggle_handle": "bracu23101281/gislr-sentences"
}''')
CONFIG_SHA = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()
CODE_SHA256 = "1551ee0d5b3f586576a7ad0934d012b79789b9a43f9eef0ac88dc6951928d9c6"
SIGNBRIDGE_COMMIT = "fd53d4f"


def find_input() -> Path:
    for depth in range(1, 6):
        hits = [p.parent for p in Path("/kaggle/input").glob("/".join(["*"] * depth) + "/test.csv")
                if (p.parent / "train.csv").is_file() and (p.parent / "test").is_dir()]
        if hits:
            assert len(hits) == 1, f"several GISLR_Stratified candidates, set INPUT_DIR: {hits}"
            return hits[0]
    raise FileNotFoundError("GISLR_Stratified not found under /kaggle/input -- add it as an input")


DATA_DIR = Path(INPUT_DIR) if INPUT_DIR else find_input()
SEQ_DIR = OUT_DIR / "sequences"
ASSETS_DIR = OUT_DIR / "assets"
PLAN_PATH = OUT_DIR / "plan.json"
MANIFEST_PATH = OUT_DIR / "manifest.json"
for d in (SEQ_DIR, ASSETS_DIR):
    d.mkdir(parents=True, exist_ok=True)


def write_json(path: Path, obj) -> None:
    tmp = path.with_suffix(".tmp")
    tmp.write_text(json.dumps(obj, indent=1), encoding="utf-8")
    os.replace(tmp, path)


print(f"input:   {DATA_DIR}")
print(f"output:  {OUT_DIR}  ({STORAGE_DTYPE}, {N_WORKERS} workers)")
print(f"config:  sha256={CONFIG_SHA[:12]}   code: sha256={CODE_SHA256[:12]} (commit {SIGNBRIDGE_COMMIT})")

## Embedded signbridge code

The exact source of the five modules the build uses, registered under their
real names, so `compose`'s own `from sb.core.schema import ...` lines resolve
unchanged. They are numpy/stdlib-only. Then the committed corpus files are
written to a temp directory, and `corpus.CORPUS_DIR` is pointed at it.

In [ ]:
# ============================================================
# Register the embedded modules + write the corpus files
# ============================================================
_SOURCES = {}
_SOURCES['sb.core.schema'] = r'''"""The landmark-tensor contract between extraction (stage 1) and training (stage 2).

This is the interface both datasets and both pipeline stages agree on, and until
now it was a paragraph of README prose plus the same 543-row layout restated in
``extraction.GROUP_LAYOUT``, ``quality.GROUPS`` and ``subsets``' docstring. It is
load-bearing for a cross-dataset claim — POPSIGN's extracted npz is written in
*GISLR holistic row order* precisely so that ``subsets.py`` indices apply to it
unchanged — and nothing checked it (TODO §9.4).

Row layout (543 rows/frame, xyz each)::

    rows   0-467  face      (468 face-mesh landmarks; row == face-mesh index)
    rows 468-488  left_hand (21)
    rows 489-521  pose      (33; holistic row = 489 + pose index)
    rows 522-542  right_hand(21)

On-disk unit (one video, POPSIGN extraction)::

    <root>/data/raw/popsign/<split>/<label>/<video_id>.npz
        landmarks   (T, 543, 3) float16, NaN where undetected
        fps         float32 scalar
        num_frames  int32 scalar   == landmarks.shape[0]

The label is carried by the path, not stored inside the file.

**NaN is data, not corruption.** An undetected group is NaN for that frame, and
that is what ``quality.py``'s detection-rate proxies measure and what the
1st-place feature pipeline normalizes over. A validator that rejected NaN would
reject every real file.

Cheap by design: :func:`validate_tensor` does O(1) structural checks by default,
so it can sit on the extraction write path for 30k+ videos (measured: 0.3 us a
call). The value scan (infinities) is opt-in and used by the npz reader, which
is a diagnostic path.
"""

from dataclasses import dataclass
from pathlib import Path

import numpy as np

SPEC_VERSION = 1

N_LANDMARKS = 543  # holistic rows per frame
N_COORDS = 3  # x, y, z
STORAGE_DTYPE = np.float16  # what extraction writes
MISSING_VALUE = "nan"  # how an undetected landmark is represented


@dataclass(frozen=True)
class LandmarkGroup:
    """One contiguous block of holistic rows."""

    name: str
    offset: int
    size: int
    result_attr: str  # the MediaPipe HolisticLandmarkerResult attribute

    @property
    def slice(self) -> slice:
        return slice(self.offset, self.offset + self.size)


GROUPS: tuple[LandmarkGroup, ...] = (
    LandmarkGroup("face", 0, 468, "face_landmarks"),
    LandmarkGroup("left_hand", 468, 21, "left_hand_landmarks"),
    LandmarkGroup("pose", 489, 33, "pose_landmarks"),
    LandmarkGroup("right_hand", 522, 21, "right_hand_landmarks"),
)

GROUP_SLICES: dict[str, slice] = {g.name: g.slice for g in GROUPS}
POSE_OFFSET: int = GROUP_SLICES["pose"].start  # holistic row of pose landmark 0

# npz keys every extracted file carries, and the dtype each must have
NPZ_KEYS: dict[str, type] = {
    "landmarks": np.float16,
    "fps": np.float32,
    "num_frames": np.int32,
}

assert sum(g.size for g in GROUPS) == N_LANDMARKS, "group sizes must tile the tensor"
assert [g.offset for g in GROUPS] == list(
    np.cumsum([0, *(g.size for g in GROUPS[:-1])])
), "groups must be contiguous and in order"


class SpecError(ValueError):
    """A landmark tensor or npz violates LANDMARK_TENSOR_V1."""


def spec() -> dict:
    """The contract as data — for embedding in a manifest or a run record."""
    return {
        "spec": "LANDMARK_TENSOR",
        "version": SPEC_VERSION,
        "n_landmarks": N_LANDMARKS,
        "n_coords": N_COORDS,
        "storage_dtype": np.dtype(STORAGE_DTYPE).name,
        "missing_value": MISSING_VALUE,
        "row_order": "gislr-holistic",
        "groups": [
            {"name": g.name, "offset": g.offset, "size": g.size} for g in GROUPS
        ],
        "npz_keys": {k: np.dtype(v).name for k, v in NPZ_KEYS.items()},
    }


def validate_tensor(
    arr: np.ndarray,
    *,
    dtype: type | None = STORAGE_DTYPE,
    check_values: bool = False,
    where: str = "landmarks",
) -> np.ndarray:
    """Structural check of a ``(T, 543, 3)`` landmark tensor; returns it.

    ``dtype=None`` accepts any floating type — loaders legitimately widen
    float16 to float32. ``check_values=True`` additionally scans for infinities;
    it is O(size), so the extraction write path leaves it off.

    T == 0 is valid: a video whose frames all failed to decode is a real,
    recorded outcome, not a malformed file.
    """
    if not isinstance(arr, np.ndarray):
        raise SpecError(f"{where}: expected ndarray, got {type(arr).__name__}")
    if arr.ndim != 3:
        raise SpecError(f"{where}: expected (T, {N_LANDMARKS}, {N_COORDS}), got {arr.shape}")
    if arr.shape[1] != N_LANDMARKS or arr.shape[2] != N_COORDS:
        raise SpecError(
            f"{where}: expected (T, {N_LANDMARKS}, {N_COORDS}), got {arr.shape} — "
            "row order is GISLR holistic (face 0-467, left_hand 468-488, "
            "pose 489-521, right_hand 522-542); a different row count means the "
            "subsets.py indices no longer apply to this file"
        )
    if dtype is not None and arr.dtype != np.dtype(dtype):
        raise SpecError(f"{where}: expected dtype {np.dtype(dtype).name}, got {arr.dtype}")
    if dtype is None and arr.dtype.kind != "f":
        raise SpecError(f"{where}: expected a floating dtype, got {arr.dtype}")
    if check_values and arr.size and np.isinf(arr).any():
        # inf cannot come out of the pipeline and silently destroys any
        # normalization downstream. An all-NaN tensor, by contrast, is a real
        # extraction outcome (nothing detected in that clip) and is a QUALITY
        # signal for quality.py to score, never a spec violation.
        raise SpecError(f"{where}: contains infinities (undetected must be NaN)")
    return arr


def validate_npz(path: Path | str, *, check_values: bool = True) -> dict:
    """Validate one extracted npz against the spec; returns a small summary.

    Reads `landmarks` fully, so this is a diagnostic/verification path rather
    than something to call per-sample in a training loop.
    """
    path = Path(path)
    with np.load(path) as d:
        missing = [k for k in NPZ_KEYS if k not in d]
        if missing:
            raise SpecError(f"{path.name}: missing npz key(s) {missing}")
        arr = d["landmarks"]
        fps = d["fps"]
        num_frames = d["num_frames"]
        validate_tensor(arr, check_values=check_values, where=path.name)
        for key, want in (("fps", np.float32), ("num_frames", np.int32)):
            got = {"fps": fps, "num_frames": num_frames}[key]
            if np.dtype(got.dtype) != np.dtype(want):
                raise SpecError(
                    f"{path.name}: {key} dtype {got.dtype}, expected {np.dtype(want).name}")
        if int(num_frames) != arr.shape[0]:
            raise SpecError(
                f"{path.name}: num_frames={int(num_frames)} but landmarks hold "
                f"{arr.shape[0]} frames")
        detected = {
            g.name: float(1.0 - np.isnan(arr[:, g.slice, 0]).all(axis=1).mean())
            if arr.shape[0] else 0.0
            for g in GROUPS
        }
    return {
        "path": str(path),
        "n_frames": int(num_frames),
        "fps": float(fps),
        "dtype": str(arr.dtype),
        "detection_rate": detected,
    }
'''
_SOURCES['sb.core.vocab'] = r'''"""The gloss vocabulary: sign name <-> class index.

Small, but it belongs in `sb-core` rather than in the recognizer, because every
direction needs the same mapping: recognition emits indices, the rescoring layer
reasons over names, and synthesis will take a name and produce a pose sequence.
A second copy of the map is a second chance for the label spaces to disagree.

GISLR ships the map as `sign_to_prediction_index_map.json` (250 classes), which
is authoritative — the competition's own indices. A dataset without one gets a
map derived deterministically from its sorted label set, so the index is stable
across machines.
"""

import json
from pathlib import Path

LABEL_MAP_FILE = "sign_to_prediction_index_map.json"


def load_label_map(data_dir: Path, filename: str = LABEL_MAP_FILE) -> dict[str, int]:
    """`{sign: index}` from a dataset's official label map."""
    return json.loads((Path(data_dir) / filename).read_text(encoding="utf-8"))


def derive_label_map(signs) -> dict[str, int]:
    """`{sign: index}` from an arbitrary label set — sorted, so two machines
    building it from the same data agree."""
    return {sign: i for i, sign in enumerate(sorted(set(signs)))}


def invert(sign2idx: dict[str, int]) -> dict[int, str]:
    """`{index: sign}` — what a prediction gets rendered through."""
    return {v: k for k, v in sign2idx.items()}


def check_covers(sign2idx: dict[str, int], signs) -> None:
    """Fail loudly when a split contains a sign the map does not know.

    Silently dropping such a row would shift every later index, which is the
    kind of corruption that surfaces as a mysterious accuracy drop.
    """
    missing = sorted(set(signs) - set(sign2idx))
    if missing:
        raise KeyError(f"signs missing from the label map: {missing}")
'''
_SOURCES['sb.recognize.features.gislr_stratified'] = r'''"""On-disk read for the GISLR_Stratified npz dataset — shared by both feature
pipelines (``base_v1`` zeroes NaN, ``firstplace_v1`` preserves it, ``evaluate``
reads it directly), so the layout fix below lives in one place.

**Row-order mismatch.** The producing notebook wrote ``landmarks`` as
``pose[0:33], face[33:501], left_hand[501:522], right_hand[522:543]`` — NOT
this repo's canonical gislr-holistic order (``sb.core.schema.GROUPS``:
``face[0:468], left_hand[468:489], pose[489:522], right_hand[522:543]``) that
every ``sb.core.subsets`` index array (ME_126, FP_118, ...) is defined
against. Reading the file as-is would silently hand a subset the wrong
physical landmarks. ``CANONICAL_TO_STORED`` is the fixed permutation that
undoes this at read time, so callers keep indexing with canonical subset
arrays and get back the landmarks those indices actually name.
"""

from pathlib import Path

import numpy as np

from sb.core.schema import N_LANDMARKS, validate_tensor

_COORD_INDEX = {"x": 0, "y": 1, "z": 2}

CANONICAL_TO_STORED = np.empty(N_LANDMARKS, dtype=np.int64)
CANONICAL_TO_STORED[0:468] = np.arange(33, 501)  # face
CANONICAL_TO_STORED[468:489] = np.arange(501, 522)  # left_hand
CANONICAL_TO_STORED[489:522] = np.arange(0, 33)  # pose
CANONICAL_TO_STORED[522:543] = np.arange(522, 543)  # right_hand (already aligned)


def load_npz(path: Path | str, rows: np.ndarray, coords: str) -> np.ndarray:
    """One GISLR_Stratified npz -> ``(T, len(rows), len(coords))`` float32.

    ``rows`` are canonical gislr-holistic indices (a subset's ``.array``);
    they're remapped onto the npz's own on-disk order before selecting. NaN is
    preserved — callers decide whether to zero it (``base_v1`` does,
    ``firstplace_v1`` must not).
    """
    path = Path(path)
    with np.load(path) as d:
        arr = d["landmarks"]
    validate_tensor(arr, dtype=None, where=path.name)
    mapped_rows = CANONICAL_TO_STORED[np.asarray(rows)]
    coord_idx = [_COORD_INDEX[c] for c in coords]
    return arr[:, mapped_rows, :][:, :, coord_idx].astype(np.float32)
'''
_SOURCES['sb.recognize.sequences.corpus'] = r'''"""The sentence corpus: ASL-gloss-order sentences over the 250 GISLR glosses.

Committed and versioned beside this module (``corpus_data/``), like
``sb.rescore``'s prompts: a dataset built from ``sentences.v1`` records
:func:`corpus_sha256` so an edited corpus can never masquerade as the one a
dataset was built from. A new wording is a new version directory, never an
in-place edit of a published one.

Layout of one version::

    corpus_data/lexicon.v1.json         {gloss: [part-of-speech tags]}
    corpus_data/sentences.v1/<theme>.txt  one sentence per line, glosses separated
                                     by spaces, '#' starts a comment

Sentence ids are ``<theme>-<line index within theme>`` so adding a line to one
theme file never renumbers another theme's sentences.
"""

from __future__ import annotations

import hashlib
import json
from collections import Counter
from dataclasses import dataclass
from pathlib import Path

CORPUS_DIR = Path(__file__).parent / "corpus_data"

POS_TAGS = (
    "noun", "verb", "adj", "time", "wh", "pronoun", "neg", "modal",
    "prep", "conj", "quant", "greeting", "response",
)


@dataclass(frozen=True)
class Sentence:
    id: str
    theme: str
    glosses: tuple[str, ...]

    def __len__(self) -> int:
        return len(self.glosses)


def lexicon_path(version: str = "v1") -> Path:
    return CORPUS_DIR / f"lexicon.{version}.json"


def sentences_dir(version: str = "v1") -> Path:
    return CORPUS_DIR / f"sentences.{version}"


def load_lexicon(version: str = "v1") -> dict[str, list[str]]:
    """``{gloss: [pos, ...]}`` -- a gloss can carry several tags (``drink``
    is noun and verb)."""
    lex = json.loads(lexicon_path(version).read_text(encoding="utf-8"))
    bad = {g: t for g, t in lex.items() if set(t) - set(POS_TAGS)}
    if bad:
        raise ValueError(f"unknown POS tags in lexicon.{version}: {bad}")
    return lex


def load_sentences(version: str = "v1") -> list[Sentence]:
    """Every sentence of one corpus version, themes in sorted-filename order."""
    out = []
    for path in sorted(sentences_dir(version).glob("*.txt")):
        theme = path.stem
        i = 0
        for line in path.read_text(encoding="utf-8").splitlines():
            line = line.split("#", 1)[0].strip()
            if not line:
                continue
            out.append(Sentence(f"{theme}-{i:03d}", theme, tuple(line.split())))
            i += 1
    return out


def corpus_sha256(version: str = "v1") -> str:
    """Content hash over the lexicon and every sentence file (name + bytes),
    recorded by anything built from the corpus. Line endings are normalized
    first: git's autocrlf rewrites them per checkout, and the same corpus
    must hash the same on every machine."""
    h = hashlib.sha256()
    files = [lexicon_path(version), *sorted(sentences_dir(version).glob("*.txt"))]
    for path in files:
        h.update(path.name.encode())
        h.update(path.read_bytes().replace(b"\r\n", b"\n"))
    return h.hexdigest()


def validate(
    sentences: list[Sentence],
    vocab,
    *,
    min_len: int = 2,
    max_len: int = 8,
    min_per_gloss: int = 12,
) -> dict:
    """Hard failures raise; soft coverage gaps are returned for the notebook
    to print. Hard: a token outside the vocabulary, a sentence outside
    ``[min_len, max_len]``, or an exact duplicate sentence (it would silently
    double that sentence's sampling weight). Soft: glosses appearing in fewer
    than ``min_per_gloss`` sentences -- the composer can still use every clip
    of such a gloss, but only by repeating its few sentences."""
    vocab = set(vocab)
    unknown = [(s.id, g) for s in sentences for g in s.glosses if g not in vocab]
    if unknown:
        raise ValueError(f"{len(unknown)} tokens outside the vocabulary, e.g. {unknown[:10]}")
    bad_len = [s.id for s in sentences if not min_len <= len(s) <= max_len]
    if bad_len:
        raise ValueError(f"{len(bad_len)} sentences outside [{min_len}, {max_len}]: {bad_len[:10]}")
    seen: dict[tuple[str, ...], str] = {}
    dups = []
    for s in sentences:
        if s.glosses in seen:
            dups.append((seen[s.glosses], s.id))
        seen.setdefault(s.glosses, s.id)
    if dups:
        raise ValueError(f"{len(dups)} duplicate sentences: {dups[:10]}")

    # coverage = number of SENTENCES containing each gloss (not token count)
    coverage = Counter(g for s in sentences for g in set(s.glosses))
    missing = sorted(vocab - set(coverage))
    thin = sorted((g, coverage[g]) for g in vocab if 0 < coverage[g] < min_per_gloss)
    lengths = Counter(len(s) for s in sentences)
    return {
        "n_sentences": len(sentences),
        "n_themes": len({s.theme for s in sentences}),
        "coverage": dict(coverage),
        "missing": missing,
        "thin": thin,
        "length_hist": dict(sorted(lengths.items())),
        "mean_len": sum(len(s) for s in sentences) / max(len(sentences), 1),
    }
'''
_SOURCES['sb.recognize.sequences.compose'] = r'''"""Sentences -> continuous landmark sequences with per-frame labels.

Two phases, deliberately split so the expensive one is resumable and the
cheap one is reviewable before anything is written:

1. **Plan** (:func:`plan_sentences`, :func:`plan_control`) -- pure
   bookkeeping over ``test.csv``: which clip fills which slot of which
   sequence, plus every synthesized gap/rest length. A plan row fully
   determines its sequence, so materialization is deterministic and can be
   interrupted and resumed per sequence.
2. **Materialize** (:func:`materialize`) -- read the clips, synthesize the
   non-sign frames, return the arrays for one sequence.

**Clip usage.** ``plan_sentences`` is coverage-driven: it keeps drawing
sentences until every source clip has been placed at least once, preferring
sentences whose glosses the chosen signer still has *unused* clips for. A
slot is only filled with an already-used clip when no sentence can be
completed from unused ones -- the ``reused`` flag records each such slot so
the reuse rate is measurable, never hidden. ``plan_control`` uses every clip
exactly once, in random order (no sentence structure): the "no language
prior" control.

**One signer per sequence.** Every slot of a sequence comes from the same
``participant_id`` -- a real continuous signer does not change identity (and
body position) at every sign boundary. A sentence is only eligible for a
signer who recorded every gloss in it.

**Synthesized frames** (GISLR clips are trimmed to the sign; nothing in the
dataset shows a hand at rest or moving between signs):

- *transition* between consecutive signs: ``g`` frames linearly interpolated,
  per landmark and coordinate, from the last frame of one clip to the first
  frame of the next. NaN on either end stays NaN (a hand absent at either
  end is not invented).
- *rest* before the first and after the last sign: the adjacent clip's edge
  frame held still with small Gaussian jitter, **both hands NaN** -- what
  MediaPipe emits while the hands are down and out of frame. The body and
  face are present.

Both are labeled null (``NULL_LABEL``) and tagged in ``frame_kind`` so an
evaluation can drop them (``frame_kind == SIGN`` recovers the hard-cut,
back-to-back concatenation exactly) or score them separately.

Landmarks are written in the repo's canonical gislr-holistic row order
(``sb.core.schema.GROUPS``), NOT the source files' stored order -- the
permutation is undone once here, so readers of this dataset never need
``CANONICAL_TO_STORED``.
"""

from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

from sb.core.schema import GROUPS, N_LANDMARKS, validate_tensor
from sb.recognize.features.gislr_stratified import load_npz
from sb.recognize.sequences.corpus import Sentence

NULL_LABEL = -1
SIGN, TRANSITION, REST = 0, 1, 2  # frame_kind values
FRAME_KINDS = {SIGN: "sign", TRANSITION: "transition", REST: "rest"}

_ALL_ROWS = np.arange(N_LANDMARKS)
HAND_ROWS = np.concatenate([
    np.arange(g.offset, g.offset + g.size) for g in GROUPS if g.name in ("left_hand", "right_hand")
])


# ---------------------------------------------------------------------------
# plan
# ---------------------------------------------------------------------------

def _synth_lengths(n_signs: int, rng: np.random.Generator, gap_range, rest_range) -> dict:
    lo, hi = gap_range
    rlo, rhi = rest_range
    return {
        "gaps": [int(g) for g in rng.integers(lo, hi + 1, size=max(n_signs - 1, 0))],
        "rest_in": int(rng.integers(rlo, rhi + 1)),
        "rest_out": int(rng.integers(rlo, rhi + 1)),
    }


def plan_sentences(
    df: pd.DataFrame,
    sentences: list[Sentence],
    rng: np.random.Generator,
    *,
    gap_range: tuple[int, int],
    rest_range: tuple[int, int],
    reuse_penalty: float = 2.0,
    repeat_penalty: float = 0.15,
    length_bonus: float = 0.0,
    pressure_weight: float = 0.0,
    progress=None,
) -> tuple[list[dict], dict]:
    """Coverage-driven sentence instantiation over every clip in ``df``.

    Loop: take the signer with the most unused clips left; score every
    sentence eligible for that signer as

        fresh_frac - reuse_penalty * reuse_frac - repeat_penalty * times_used_by_signer
        + length_bonus * n_fresh / max_len + pressure_weight * pressure

    where ``fresh_frac`` is the fraction of the sentence's slots fillable with
    that signer's unused clips and ``pressure`` is the mean number of unused
    clips its glosses still have, relative to the signer's average; take the
    best (random tiebreak), fill it. Stops when every clip is placed. Returns
    ``(plan, stats)``.

    Scoring by *fraction* keeps any fully-fresh sentence ahead of a partly
    reused one; ``length_bonus`` then breaks the tie toward longer sentences
    (without it, 2-gloss sentences win because they are the easiest to fill
    fresh). ``pressure`` drains the glosses a signer has most of first, so the
    last clips left are not a scatter no sentence can use without reuse.
    ``repeat_penalty`` spreads a signer across many sentences instead of
    repeating a favourite.
    """
    glosses = sorted(df["sign"].unique())
    gidx = {g: i for i, g in enumerate(glosses)}
    G, N = len(glosses), len(sentences)
    S = np.zeros((N, G), dtype=np.int32)
    for i, s in enumerate(sentences):
        for g in s.glosses:
            S[i, gidx[g]] += 1
    lens = S.sum(1).astype(np.float64)
    max_len = lens.max()

    unused: dict[int, list[list[str]]] = {}
    used: dict[int, list[list[str]]] = {}
    for pid in sorted(int(p) for p in df["participant_id"].unique()):
        sub = df[df["participant_id"] == pid]
        pool = [[] for _ in range(G)]
        for uid, g in zip(sub["uid"], sub["sign"]):
            pool[gidx[g]].append(uid)
        for lst in pool:
            rng.shuffle(lst)
        unused[pid] = pool
        used[pid] = [[] for _ in range(G)]
    avail = {p: np.array([len(x) for x in pool]) for p, pool in unused.items()}
    has = {p: a > 0 for p, a in avail.items()}
    eligible = {p: (S > 0).astype(np.int32) @ (~h).astype(np.int32) == 0 for p, h in has.items()}
    uses = {p: np.zeros(N) for p in unused}

    plan, orphans = [], 0
    total = remaining = len(df)
    while remaining > 0:
        p = max(avail, key=lambda q: avail[q].sum())
        a = avail[p]
        fresh = np.minimum(S, a[None, :]).sum(1)
        score = (fresh - reuse_penalty * (lens - fresh)) / lens - repeat_penalty * uses[p]
        score += length_bonus * fresh / max_len
        if pressure_weight:
            pressure = (S @ a) / lens / max(a[a > 0].mean(), 1e-9)
            score += pressure_weight * pressure
        score += rng.random(N) * 1e-6
        score[~eligible[p] | (fresh == 0)] = -np.inf
        if not np.isfinite(score).any():
            # this signer's leftover clips are of glosses no eligible sentence
            # contains -- they cannot be placed in a sentence at all
            orphans += int(a.sum())
            remaining -= int(a.sum())
            avail[p] = np.zeros_like(a)
            continue
        k = int(np.argmax(score))
        uses[p][k] += 1
        uids, reused = [], []
        for g in sentences[k].glosses:
            j = gidx[g]
            if unused[p][j]:
                uid = unused[p][j].pop()
                used[p][j].append(uid)
                a[j] -= 1
                remaining -= 1
                uids.append(uid)
                reused.append(False)
            else:
                uids.append(used[p][j][int(rng.integers(len(used[p][j])))])
                reused.append(True)
        plan.append({
            "kind": "sentence",
            "sentence_id": sentences[k].id,
            "participant_id": int(p),
            "glosses": list(sentences[k].glosses),
            "source_uids": uids,
            "reused": reused,
            **_synth_lengths(len(uids), rng, gap_range, rest_range),
        })
        if progress is not None:
            progress(total - remaining, total)

    n_slots = sum(len(r["source_uids"]) for r in plan)
    n_reused = sum(sum(r["reused"]) for r in plan)
    stats = {
        "n_sequences": len(plan),
        "n_slots": n_slots,
        "n_reused_slots": n_reused,
        "reuse_rate": n_reused / max(n_slots, 1),
        "n_clips": total,
        "n_orphan_clips": orphans,
        "n_distinct_sentences": len({r["sentence_id"] for r in plan}),
    }
    return plan, stats


def plan_control(
    df: pd.DataFrame,
    lengths: np.ndarray,
    rng: np.random.Generator,
    *,
    gap_range: tuple[int, int],
    rest_range: tuple[int, int],
) -> list[dict]:
    """Every clip exactly once, random gloss order, one signer per sequence;
    sequence lengths drawn from ``lengths`` (pass the sentence split's
    lengths so the two splits differ only in word order/meaning)."""
    plan = []
    for pid in sorted(int(p) for p in df["participant_id"].unique()):
        sub = df[df["participant_id"] == pid]
        idx = rng.permutation(len(sub))
        uids = sub["uid"].to_numpy()[idx]
        signs = sub["sign"].to_numpy()[idx]
        i = 0
        while i < len(uids):
            n = int(rng.choice(lengths))
            if len(uids) - (i + n) == 1:  # never leave a 1-sign tail
                n += 1
            j = min(i + n, len(uids))
            plan.append({
                "kind": "control",
                "sentence_id": "",
                "participant_id": int(pid),
                "glosses": [str(s) for s in signs[i:j]],
                "source_uids": [str(u) for u in uids[i:j]],
                "reused": [False] * (j - i),
                **_synth_lengths(j - i, rng, gap_range, rest_range),
            })
            i = j
    order = rng.permutation(len(plan))  # interleave signers
    return [plan[i] for i in order]


# ---------------------------------------------------------------------------
# materialize
# ---------------------------------------------------------------------------

def load_clip(data_dir: Path, relpath: str) -> np.ndarray:
    """One GISLR_Stratified clip in canonical row order, NaN preserved."""
    return load_npz(Path(data_dir) / relpath, _ALL_ROWS, "xyz")


def transition(a: np.ndarray, b: np.ndarray, n: int) -> np.ndarray:
    """``n`` frames strictly between frame ``a`` and frame ``b`` (endpoints
    excluded); NaN at either end propagates."""
    t = (np.arange(1, n + 1, dtype=np.float32) / (n + 1))[:, None, None]
    return a[None] + t * (b - a)[None]


def rest(edge: np.ndarray, n: int, rng: np.random.Generator, jitter: float) -> np.ndarray:
    """``n`` frames of ``edge`` held still with Gaussian jitter, hands NaN."""
    out = np.repeat(edge[None], n, axis=0).astype(np.float32)
    out += rng.normal(0.0, jitter, size=out.shape).astype(np.float32)
    out[:, HAND_ROWS, :] = np.nan
    return out


def materialize(
    row: dict,
    data_dir: Path,
    relpath_of: dict[str, str],
    label_of: dict[str, int],
    rng: np.random.Generator,
    *,
    rest_jitter: float,
) -> dict:
    """One plan row -> ``{landmarks, frame_labels, frame_kind, segments, labels}``.

    ``segments[i] = (start, end)``, end exclusive, in output frames: slicing
    ``landmarks[start:end]`` returns source clip ``i`` bit-for-bit.
    """
    clips = [load_clip(data_dir, relpath_of[u]) for u in row["source_uids"]]
    labels = [label_of[g] for g in row["glosses"]]
    parts, kinds, flabels, segments = [], [], [], []
    t = 0

    def add(arr, kind, label):
        nonlocal t
        parts.append(arr)
        kinds.append(np.full(len(arr), kind, np.uint8))
        flabels.append(np.full(len(arr), label, np.int16))
        t += len(arr)

    add(rest(clips[0][0], row["rest_in"], rng, rest_jitter), REST, NULL_LABEL)
    for i, (clip, lab) in enumerate(zip(clips, labels)):
        if i > 0:
            add(transition(clips[i - 1][-1], clip[0], row["gaps"][i - 1]), TRANSITION, NULL_LABEL)
        segments.append((t, t + len(clip)))
        add(clip, SIGN, lab)
    add(rest(clips[-1][-1], row["rest_out"], rng, rest_jitter), REST, NULL_LABEL)

    landmarks = np.concatenate(parts).astype(np.float32)
    validate_tensor(landmarks, dtype=np.float32, check_values=True, where="sequence")
    return {
        "landmarks": landmarks,
        "frame_labels": np.concatenate(flabels),
        "frame_kind": np.concatenate(kinds),
        "segments": np.asarray(segments, dtype=np.int32).reshape(-1, 2),
        "labels": np.asarray(labels, dtype=np.int16),
    }


def write_sequence(path: Path, arrays: dict, dtype=np.float32) -> Path:
    """Atomic compressed npz (temp file + ``os.replace``). ``dtype`` is the
    stored landmark precision: float32 (default) keeps every segment
    bit-identical to its float32 source; float16 halves the size."""
    arrays = {**arrays, "landmarks": arrays["landmarks"].astype(dtype)}
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp.npz")
    np.savez_compressed(tmp, **arrays)
    os.replace(tmp, path)
    return path


def read_sequence(path: Path) -> dict:
    with np.load(path) as d:
        return {k: d[k] for k in d.files}


def check_roundtrip(arrays: dict, row: dict, data_dir: Path, relpath_of: dict[str, str]) -> None:
    """Every segment slices back to its exact source clip; labels and kinds
    agree with the segments. Raises on the first mismatch.

    The source is cast to the stored dtype before comparing, so a float16
    build is checked for exactly what float16 storage promises (the same
    rounding of the same clip), and a float32 build stays bit-exact."""
    fk, fl = arrays["frame_kind"], arrays["frame_labels"]
    covered = np.zeros(len(fk), bool)
    for (s, e), uid, lab in zip(arrays["segments"], row["source_uids"], arrays["labels"]):
        src = load_clip(data_dir, relpath_of[uid]).astype(arrays["landmarks"].dtype)
        if not np.array_equal(arrays["landmarks"][s:e], src, equal_nan=True):
            raise AssertionError(f"segment {s}:{e} != source clip {uid}")
        if not ((fk[s:e] == SIGN).all() and (fl[s:e] == lab).all()):
            raise AssertionError(f"labels/kinds wrong inside segment {s}:{e}")
        covered[s:e] = True
    if (fk[~covered] == SIGN).any() or (fl[~covered] != NULL_LABEL).any():
        raise AssertionError("a frame outside every segment is labelled as a sign")


def summarize_row(row: dict, arrays: dict, seq_id: str, relpath: str) -> dict:
    """The ``sequences.csv`` row for one materialized sequence."""
    seg = arrays["segments"]
    return {
        "seq_id": seq_id,
        "kind": row["kind"],
        "sentence_id": row["sentence_id"],
        "participant_id": row["participant_id"],
        "n_signs": len(row["glosses"]),
        "glosses": " ".join(row["glosses"]),
        "labels": " ".join(str(int(x)) for x in arrays["labels"]),
        "starts": " ".join(str(int(s)) for s in seg[:, 0]),
        "ends": " ".join(str(int(e)) for e in seg[:, 1]),
        "source_uids": " ".join(row["source_uids"]),
        "reused": " ".join(str(int(r)) for r in row["reused"]),
        "gaps": " ".join(str(g) for g in row["gaps"]),
        "rest_in": row["rest_in"],
        "rest_out": row["rest_out"],
        "n_frames": int(len(arrays["frame_kind"])),
        "npz_relpath": relpath,
    }
'''


def _register(name: str, source: str) -> types.ModuleType:
    parts = name.split(".")
    for i in range(1, len(parts)):  # namespace parents: sb, sb.core, ...
        parent = ".".join(parts[:i])
        if parent not in sys.modules:
            pkg = types.ModuleType(parent)
            pkg.__path__ = []
            sys.modules[parent] = pkg
    module = types.ModuleType(name)
    module.__file__ = f"<embedded {name}>"
    sys.modules[name] = module
    exec(compile(source, module.__file__, "exec"), module.__dict__)
    setattr(sys.modules[".".join(parts[:-1])], parts[-1], module)
    return module


for _name, _src in _SOURCES.items():
    _register(_name, _src)
assert hashlib.sha256("".join(_SOURCES.values()).encode()).hexdigest() == CODE_SHA256

compose = sys.modules["sb.recognize.sequences.compose"]
corpus = sys.modules["sb.recognize.sequences.corpus"]

_CORPUS_FILES = {}
_CORPUS_FILES['lexicon.v1.json'] = r'''{
  "TV": ["noun"],
  "after": ["time", "prep"],
  "airplane": ["noun"],
  "all": ["quant"],
  "alligator": ["noun"],
  "animal": ["noun"],
  "another": ["quant"],
  "any": ["quant"],
  "apple": ["noun"],
  "arm": ["noun"],
  "aunt": ["noun"],
  "awake": ["adj"],
  "backyard": ["noun"],
  "bad": ["adj"],
  "balloon": ["noun"],
  "bath": ["noun", "verb"],
  "because": ["conj"],
  "bed": ["noun"],
  "bedroom": ["noun"],
  "bee": ["noun"],
  "before": ["time", "prep"],
  "beside": ["prep"],
  "better": ["adj"],
  "bird": ["noun"],
  "black": ["adj"],
  "blow": ["verb"],
  "blue": ["adj"],
  "boat": ["noun"],
  "book": ["noun"],
  "boy": ["noun"],
  "brother": ["noun"],
  "brown": ["adj"],
  "bug": ["noun"],
  "bye": ["greeting"],
  "callonphone": ["verb"],
  "can": ["modal"],
  "car": ["noun"],
  "carrot": ["noun"],
  "cat": ["noun"],
  "cereal": ["noun"],
  "chair": ["noun"],
  "cheek": ["noun"],
  "child": ["noun"],
  "chin": ["noun"],
  "chocolate": ["noun"],
  "clean": ["verb", "adj"],
  "close": ["verb"],
  "closet": ["noun"],
  "cloud": ["noun"],
  "clown": ["noun"],
  "cow": ["noun"],
  "cowboy": ["noun"],
  "cry": ["verb"],
  "cut": ["verb"],
  "cute": ["adj"],
  "dad": ["noun"],
  "dance": ["verb"],
  "dirty": ["adj"],
  "dog": ["noun"],
  "doll": ["noun"],
  "donkey": ["noun"],
  "down": ["prep"],
  "drawer": ["noun"],
  "drink": ["verb", "noun"],
  "drop": ["verb"],
  "dry": ["adj", "verb"],
  "dryer": ["noun"],
  "duck": ["noun"],
  "ear": ["noun"],
  "elephant": ["noun"],
  "empty": ["adj"],
  "every": ["quant"],
  "eye": ["noun"],
  "face": ["noun"],
  "fall": ["verb"],
  "farm": ["noun"],
  "fast": ["adj"],
  "feet": ["noun"],
  "find": ["verb"],
  "fine": ["adj"],
  "finger": ["noun"],
  "finish": ["verb"],
  "fireman": ["noun"],
  "first": ["time", "adj"],
  "fish": ["noun"],
  "flag": ["noun"],
  "flower": ["noun"],
  "food": ["noun"],
  "for": ["prep"],
  "frenchfries": ["noun"],
  "frog": ["noun"],
  "garbage": ["noun"],
  "gift": ["noun"],
  "giraffe": ["noun"],
  "girl": ["noun"],
  "give": ["verb"],
  "glasswindow": ["noun"],
  "go": ["verb"],
  "goose": ["noun"],
  "grandma": ["noun"],
  "grandpa": ["noun"],
  "grass": ["noun"],
  "green": ["adj"],
  "gum": ["noun"],
  "hair": ["noun"],
  "happy": ["adj"],
  "hat": ["noun"],
  "hate": ["verb"],
  "have": ["verb"],
  "haveto": ["modal"],
  "head": ["noun"],
  "hear": ["verb"],
  "helicopter": ["noun"],
  "hello": ["greeting"],
  "hen": ["noun"],
  "hesheit": ["pronoun"],
  "hide": ["verb"],
  "high": ["adj"],
  "home": ["noun"],
  "horse": ["noun"],
  "hot": ["adj"],
  "hungry": ["adj"],
  "icecream": ["noun"],
  "if": ["conj"],
  "into": ["prep"],
  "jacket": ["noun"],
  "jeans": ["noun"],
  "jump": ["verb"],
  "kiss": ["verb"],
  "kitty": ["noun"],
  "lamp": ["noun"],
  "later": ["time"],
  "like": ["verb"],
  "lion": ["noun"],
  "lips": ["noun"],
  "listen": ["verb"],
  "look": ["verb"],
  "loud": ["adj"],
  "mad": ["adj"],
  "make": ["verb"],
  "man": ["noun"],
  "many": ["quant"],
  "milk": ["noun"],
  "minemy": ["pronoun"],
  "mitten": ["noun"],
  "mom": ["noun"],
  "moon": ["noun"],
  "morning": ["time"],
  "mouse": ["noun"],
  "mouth": ["noun"],
  "nap": ["noun", "verb"],
  "napkin": ["noun"],
  "night": ["time"],
  "no": ["response", "neg"],
  "noisy": ["adj"],
  "nose": ["noun"],
  "not": ["neg"],
  "now": ["time"],
  "nuts": ["noun"],
  "old": ["adj"],
  "on": ["prep"],
  "open": ["verb"],
  "orange": ["adj", "noun"],
  "outside": ["prep", "noun"],
  "owie": ["noun"],
  "owl": ["noun"],
  "pajamas": ["noun"],
  "pen": ["noun"],
  "pencil": ["noun"],
  "penny": ["noun"],
  "person": ["noun"],
  "pig": ["noun"],
  "pizza": ["noun"],
  "please": ["greeting"],
  "police": ["noun"],
  "pool": ["noun"],
  "potty": ["noun"],
  "pretend": ["verb"],
  "pretty": ["adj"],
  "puppy": ["noun"],
  "puzzle": ["noun"],
  "quiet": ["adj"],
  "radio": ["noun"],
  "rain": ["noun", "verb"],
  "read": ["verb"],
  "red": ["adj"],
  "refrigerator": ["noun"],
  "ride": ["verb"],
  "room": ["noun"],
  "sad": ["adj"],
  "same": ["adj"],
  "say": ["verb"],
  "scissors": ["noun"],
  "see": ["verb"],
  "shhh": ["greeting"],
  "shirt": ["noun"],
  "shoe": ["noun"],
  "shower": ["noun", "verb"],
  "sick": ["adj"],
  "sleep": ["verb"],
  "sleepy": ["adj"],
  "smile": ["verb"],
  "snack": ["noun"],
  "snow": ["noun", "verb"],
  "stairs": ["noun"],
  "stay": ["verb"],
  "sticky": ["adj"],
  "store": ["noun"],
  "story": ["noun"],
  "stuck": ["adj"],
  "sun": ["noun"],
  "table": ["noun"],
  "talk": ["verb"],
  "taste": ["verb"],
  "thankyou": ["greeting"],
  "that": ["pronoun"],
  "there": ["pronoun"],
  "think": ["verb"],
  "thirsty": ["adj"],
  "tiger": ["noun"],
  "time": ["noun", "wh"],
  "tomorrow": ["time"],
  "tongue": ["noun"],
  "tooth": ["noun"],
  "toothbrush": ["noun"],
  "touch": ["verb"],
  "toy": ["noun"],
  "tree": ["noun"],
  "uncle": ["noun"],
  "underwear": ["noun"],
  "up": ["prep"],
  "vacuum": ["noun", "verb"],
  "wait": ["verb"],
  "wake": ["verb"],
  "water": ["noun"],
  "wet": ["adj"],
  "weus": ["pronoun"],
  "where": ["wh"],
  "white": ["adj"],
  "who": ["wh"],
  "why": ["wh"],
  "will": ["modal", "time"],
  "wolf": ["noun"],
  "yellow": ["adj"],
  "yes": ["response"],
  "yesterday": ["time"],
  "yourself": ["pronoun"],
  "yucky": ["adj"],
  "zebra": ["noun"],
  "zipper": ["noun"]
}
'''
_CORPUS_FILES['sentences.v1/animals.txt'] = r'''# Animals, pets, farm, zoo.
minemy dog hungry
minemy dog like water
minemy dog brown white
minemy cat black
minemy cat sleep bed
minemy kitty cute
minemy kitty hide closet
minemy puppy cute brown
minemy puppy jump bed
minemy puppy dirty haveto bath
puppy like toy
puppy kiss minemy face
dog see cat
dog look cat hide tree
cat hide bedroom
kitty go up tree
kitty stuck tree
kitty stuck tree cry
fireman find kitty tree
dog find toy backyard
dog go backyard
dog stay outside night
dog noisy night
dog loud bad
dog drink water empty
cat drink milk
kitty drink milk now
cat hate water
cat hate bath
kitty like sun
cat sleep sun
cat like fish
kitty like fish food
bird up high
bird high tree
bird yellow pretty
bird blue loud morning
owl awake night
owl look moon night
owl white
owl say who
duck water
duck into pool
duck yellow cute
duck goose farm
goose loud
goose white
goose bad
hen farm
hen brown
cow farm
cow brown white
cow give milk
cow noisy morning
pig farm dirty
pig dirty like water
pig hungry
horse farm
horse brown fast
horse go fast
donkey farm
donkey old
donkey stuck
donkey hungry carrot
horse like carrot
horse like apple
mouse quiet
mouse hide
mouse like nuts
mouse hide drawer
mouse find food
cat look mouse
cat find mouse
frog green
frog jump high
frog jump water
frog jump pool
frog like bug
bug black
bug on flower
bug bad
bee yellow black
bee on flower
bee touch owie
bee go flower
fish water
fish orange
fish blue pretty
fish go fast water
lion mad
lion loud
lion hungry
tiger orange black
tiger fast
tiger stay outside
elephant old
elephant old happy
elephant nose water
elephant drink water
giraffe high
giraffe look tree high
zebra black white
zebra horse same
zebra go fast
alligator water
alligator bad
alligator mouth open
alligator tooth
wolf night
wolf loud night
wolf black
wolf hungry
animal farm many
animal outside many
animal hungry
all animal sleep night
many animal outside
yourself like animal
weus go farm see animal
weus see lion tiger
yesterday weus see elephant giraffe
tomorrow weus go farm see cow
child like zebra
boy like alligator
girl like kitty puppy
grandpa have horse farm
grandma have hen farm
uncle have donkey
aunt have bird
mom not like mouse
dad not like bug
brother not like bee
child cry because bee
child touch puppy
child kiss kitty
dog cat not same
cow pig horse farm
lion tiger zebra giraffe outside
duck goose hen farm
dog kitty stay home
where minemy dog
where minemy cat
where puppy
where kitty hide
cat where
dog where
why dog loud
why cat hide
why puppy cry
who have dog
who have kitty
that dog who
that cat hesheit minemy
dog finish food
puppy finish milk
dog bad dog
dog fine now
'''
_CORPUS_FILES['sentences.v1/body.txt'] = r'''# Body, health, senses.
minemy head owie
minemy arm owie
minemy finger owie
minemy feet owie
minemy ear owie
minemy tooth owie
minemy tongue owie
minemy mouth owie
minemy eye owie
minemy cheek owie
minemy chin owie
minemy nose owie cry
child fall arm owie
child fall head owie cry
child cut finger owie
bee touch arm owie
owie where
where owie
owie better now
owie finish
mom kiss owie better
grandma kiss owie
dad look owie finger
minemy sick
minemy sick haveto stay home
child sick hot head
child sick stay bed
child sick not hungry
child sick not go outside
sick child sleep all morning
mom sick dad make food
yesterday minemy sick now fine
yesterday child sick now better
minemy fine thankyou
yourself fine
yourself sick
yourself better now
yourself feet dirty
yourself face dirty
yourself hair wet
face dirty haveto clean
face wet
face happy
face sad
face red hot
clean face morning
hair wet
hair dry
hair dirty haveto shower
hair brown pretty
mom cut hair
dad cut hair brother
grandma cut minemy hair
girl hair pretty
girl hair yellow
boy hair black
grandpa hair white
grandma hair white old
feet dirty
feet wet
feet dirty backyard
feet owie
shoe feet
touch nose
touch ear
touch head
touch chin
touch cheek
touch tongue yucky
touch lips
touch arm
touch finger
eye blue
eye brown
eye green
minemy eye brown
dad eye blue
cat eye yellow
owl eye yellow
close eye sleep
open eye
open mouth
close mouth
open mouth tongue
lips red
lips dry
lips sticky
tongue red
tongue taste chocolate
tongue taste yucky
nose red
clown nose red
nose owie sick
ear hear
ear listen
hear loud noisy
hear quiet
not hear
grandpa not hear
listen please
listen mom say
see not
look see
look face
look finger
finger many
finger sticky
finger dirty
finger taste icecream
arm owie cry
arm dirty
arm wet
chin dirty food
cheek red
kiss cheek
mom kiss cheek child
grandpa kiss grandma cheek
tooth white
tooth dirty haveto toothbrush
tooth drop
tooth fall morning
tooth yellow
mouth dirty napkin
mouth open
mouth close quiet
mouth taste
'''
_CORPUS_FILES['sentences.v1/clothes.txt'] = r'''# Clothes and getting dressed.
minemy shirt red
minemy shirt blue dirty
minemy shirt white wet
minemy jeans blue
minemy jeans dirty
minemy jacket green
minemy jacket where
minemy hat yellow
minemy hat where
minemy shoe black
minemy shoe where
minemy mitten red
minemy mitten where
minemy pajamas blue
minemy underwear clean
minemy zipper stuck
zipper stuck
zipper stuck jacket
zipper jacket stuck mom
zipper jeans stuck
zipper jacket open
zipper close
close zipper jacket now
open zipper please
jacket zipper stuck
snow haveto jacket hat mitten
snow outside haveto jacket
rain outside haveto jacket
hot outside no jacket
jacket hot
jacket close
jacket closet
hat mitten jacket snow
hat head
hat on head
hat cowboy brown
hat fireman red
hat clown yellow
hat pretty
hat drop
hat drop wet
grandma make hat mitten
grandma make mitten
mitten wet
mitten snow wet
mitten red blue
mitten where find
mitten another where
mitten dryer
shoe dirty
shoe wet
shoe clean
shoe old
shoe stuck
shoe beside bed
shoe beside chair
shoe closet
shoe on feet
gum shoe
shoe gum sticky yucky
shirt dirty
shirt clean
shirt wet
shirt dry
shirt yellow
shirt orange
shirt pretty
shirt green
shirt brown
shirt old
shirt drawer
shirt dryer
shirt same
shirt jeans same blue
jeans dirty
jeans wet
jeans old
jeans drawer
jeans dryer
underwear drawer
underwear clean
underwear dirty
underwear dryer
underwear wet
child underwear wet
child underwear dry
pajamas bed
pajamas drawer
pajamas night
pajamas pretty
pajamas white
pajamas finish
pajamas morning
girl pretty shirt
boy shirt dirty
brother jeans dirty
dad jacket brown
mom jacket pretty
uncle hat black
aunt hat pretty
grandpa jacket old
cowboy jeans hat
police hat jacket blue
fireman jacket hat yellow
clown shirt red yellow
yourself jacket where
yourself shoe dirty
yourself zipper open
who have minemy hat
who have minemy shoe
where hat
where shoe
where jacket
where mitten
'''
_CORPUS_FILES['sentences.v1/conversation.txt'] = r'''# Conversation: greetings, questions, answers, conditions, quantifiers.
hello yourself fine
hello grandma yourself fine
hello there minemy puppy
hello man police
hello person there
bye bye grandpa
bye weus go home now
bye see yourself tomorrow
bye aunt thankyou gift
bye uncle see later
bye dog stay home
bye mom go store
bye kitty sleep
yes minemy like
yes please thankyou
yes minemy hungry
yes minemy finish
yes weus go pool
yes hesheit minemy brother
yes minemy can
yes that minemy toy
yes dog there
no minemy not like
no thankyou minemy not hungry
no minemy not finish
no that not minemy
no dog not there
yourself hungry yes no
yourself thirsty yes no
yourself like pizza yes no
yourself finish yes no
yourself go pool yes no
any food
any milk refrigerator
any icecream
any gum
any penny
any water
any snack please
any chocolate there
yourself have any penny
yourself have any gum
store have any apple
refrigerator have any milk
minemy not have any toy
no any food there
dog not have any water
every morning
every night
every morning minemy cereal milk
every night mom read story
every night grandpa look moon
every morning dad shower
every child have gift
every animal hungry
every flower pretty
every toy dirty
every person happy
every dog bad
every morning bird loud
if yourself hungry say
if yourself sleepy go bed
if dog hungry give food
if rain no pool
if cat hide find
if yourself finish go outside
if yourself thirsty drink water
if hot drink water
if grandma callonphone listen
for yourself
for minemy
gift for dad
gift for yourself
food for dog
milk for kitty
water for flower
carrot for horse
nuts for mouse
dog there
cat there
there tree
there car
there bird
look there
go there
who there
that minemy
that yourself
that man
that person
that car
that hat
that dog minemy
that pretty
that yucky
that bad
who that
who yourself
who hesheit
who go
who cry
who hide
who callonphone
who noisy
who finish
who drop
why cry
why hide
why mad
why noisy
why go
why not
why wait
where yourself
where go
where mom
where dad
where grandma
where toy
where book
where hesheit
hesheit go
hesheit sleep
hesheit hungry
hesheit happy
hesheit cry
hesheit mad
hesheit fine
hesheit cute
hesheit man
can go
can not
can yourself find minemy hat
can minemy go outside
can yourself hear
can yourself see
can dog jump high
can weus go pool
say please
say thankyou
say hello
say bye
say yes
say no
say bye grandma
listen minemy
listen mom
listen dad
listen story
listen radio
same same
same shirt
same hat
same car
same dog
another story
another book
another apple
another snack please
another drink
another toy
all finish
all food finish now
all go home
all sleep
all happy
all dirty
all clean
'''
_CORPUS_FILES['sentences.v1/family.txt'] = r'''# Family and people. ASL gloss order (time first, topic-comment, wh-word last
# or first). Pronouns available: minemy, hesheit, weus, yourself, that, there.
hello mom
hello dad
hello grandma
hello grandpa
bye mom
bye dad see later
bye grandma see tomorrow
hello uncle thankyou
hello aunt please
mom dad go store
dad go store now
mom callonphone grandma
grandma callonphone mom morning
grandpa callonphone dad yesterday
uncle callonphone aunt night
minemy mom happy
minemy dad sleepy now
minemy dad sleepy
minemy grandma old
minemy grandpa old happy
minemy brother mad
minemy brother fast
minemy aunt pretty
minemy uncle loud
minemy uncle like pretend clown
brother hide closet
brother hide bedroom
brother hate bath
brother like pizza
brother like frenchfries
girl like doll
girl like pretty flower
boy like toy car
boy like airplane
boy ride horse
boy ride boat
girl ride horse farm
child sleep now
child cry because hungry
child cry because sleepy
child happy because gift
child sick stay home
child dirty haveto bath
mom give child milk
dad give child apple
grandma give minemy gift
grandpa give child penny
aunt give girl doll
uncle give boy toy
mom kiss child
grandma kiss cheek
dad kiss mom
mom read story child
dad read book night
grandpa read story before sleep
grandma make cereal morning
grandma make food
mom make pizza
dad make snack
brother make puzzle
mom say please
child say thankyou
child say yes
child say no
boy say hello girl
girl say bye boy
man look car
man drink water
man find penny
person wait there
person talk loud
person callonphone police
police car fast
police find person outside
police find child
fireman go fast
fireman go up stairs fast
fireman hat red
fireman jacket yellow
clown hat red nose red
clown make balloon
clown blow balloon
clown hat
cowboy ride horse
cowboy hat brown
cowboy farm
minemy mom dad go farm
weus go grandma home
tomorrow weus go grandpa home
weus like same food
weus go outside
weus like dance
weus go pool
weus go store later
hesheit minemy brother
hesheit minemy aunt
hesheit minemy uncle
hesheit man police
who that man
who that girl
who that boy
who that person
that man who
that girl minemy aunt child
that boy minemy brother
minemy brother old
yourself go
yourself stay home
yourself wake now
yourself clean room
yourself have gift
yourself like dog
dad wake first morning
mom wake child morning
grandma wait home
grandpa sleep chair
grandpa nap chair
grandma nap after food
uncle ride car store
aunt go store
aunt look pretty
uncle hear radio
dad listen radio
mom listen child
girl smile
boy smile mom
child touch dog
child pretend fireman
child pretend police
boy pretend cowboy
girl pretend mom
mom haveto go now
dad haveto wait
child haveto sleep
brother haveto clean room
child can read
child can jump high
boy can ride fast
girl can dance
grandpa can not hear
grandma can not see
dad will go tomorrow
mom will go home later
uncle will go farm tomorrow
aunt will callonphone later
'''
_CORPUS_FILES['sentences.v1/feelings.txt'] = r'''# Feelings, thinking, quiet/noisy, likes and dislikes.
minemy sad
minemy sad because dog sick
minemy sad because grandma go home
child sad cry because balloon go up
girl sad because doll drop
boy sad because toy where
yourself sad why
why yourself sad
why child sad
sad dog wait mom
sad kitty cry night
not sad now happy
minemy happy because gift
child happy smile
girl smile cute
boy smile mom happy
child smile grandpa
grandma smile cute child
smile please
smile dad look
yourself smile pretty
why yourself smile
cute puppy smile
cute kitty
cute duck yellow
cute child
cute pig dirty
cute zebra
minemy mad
dad mad because room dirty
mom mad because child jump bed
grandpa mad dog loud
brother mad because minemy have toy
yourself mad why
why yourself mad
lion mad loud
not mad now fine
minemy think
minemy think yes
minemy think no
minemy think dog hungry
minemy think rain later
minemy think grandma sleep now
minemy think hesheit sick
think yourself sleepy
dad think car stuck
mom think child hide closet
child think mouse drawer
think first after say
think before say
yourself think why
yourself think dog where
shhh quiet please
shhh child sleep bed
shhh child sleep now
shhh grandpa nap
shhh dad talk callonphone
shhh listen
shhh TV loud
shhh no talk
shhh mom read story
quiet please
quiet now
quiet read book
child quiet
cat quiet
mouse quiet hide
home quiet night
talk loud
talk quiet please
talk later
talk now
talk grandma callonphone
dad talk uncle
mom talk aunt
child talk many
girl talk boy
no talk now
who talk
yourself talk loud
noisy child
noisy dog
noisy TV
noisy outside
noisy close glasswindow
minemy sleepy
child sleepy now
dog sleepy sun
kitty sleepy bed
grandpa sleepy chair
sleepy nap time
yourself sleepy go bed
minemy thirsty now
dog thirsty
child thirsty drink milk
hot outside thirsty
hungry thirsty
yourself owie better now
owie better
minemy better now
better go now
dog better now
pizza better frenchfries
icecream better chocolate
bad dog
bad cat
bad mouse
taste bad
dog bad dirty
minemy fine
fine thankyou
dog fine
grandma fine now
minemy hate carrot
minemy hate bath
dog hate cat
cat hate dog
brother hate shower
girl hate bug
hate rain
hate snow
minemy like yourself
minemy like dog
yourself like minemy
weus like animal
'''
_CORPUS_FILES['sentences.v1/food.txt'] = r'''# Food and drink. ASL FOOD and EAT share one sign, so "food" also stands in
# for the verb where natural.
minemy hungry
minemy thirsty
minemy thirsty drink water please
minemy hungry food now please
child hungry haveto food now
child thirsty give milk please
mom give child cereal milk morning
morning minemy like cereal milk
dad make cereal morning
cereal milk morning
cereal empty
cereal finish
milk empty refrigerator
refrigerator empty haveto go store
refrigerator open close
close refrigerator please
open refrigerator find milk
milk refrigerator
water refrigerator drink now
apple refrigerator
chocolate refrigerator
icecream refrigerator
icecream hot sun sticky
icecream sticky finger
icecream taste fine
icecream chocolate taste
minemy like chocolate icecream
brother like icecream
yourself like icecream
who like icecream
who like chocolate
who have gum
where gum
gum sticky
gum sticky hair
gum sticky shoe
gum stuck hair
gum stuck shoe yucky
minemy like gum
child not like carrot yucky
carrot yucky
carrot orange
orange orange
orange taste better apple
apple red
apple green
apple red taste
apple green yucky
grandma give apple child
dad cut apple
mom cut pizza
mom cut orange
dad cut carrot
pizza hot
pizza hot taste
minemy like pizza
weus like pizza frenchfries
frenchfries hot
frenchfries taste
frenchfries not hot yucky
brother hate carrot like frenchfries
nuts brown
mouse nuts
nuts snack
snack time
snack time now
minemy hungry snack please
yourself like snack now
yourself hungry
yourself thirsty
yourself finish food
finish food now go outside
finish milk first
first food after go outside
first milk after chocolate
food table
food hot
food not hot
food taste yucky
food taste fine
food empty
food finish
all food finish
many food table
dad make food night
mom make food morning
grandma make pizza
grandma make chocolate
grandpa like nuts
uncle like chocolate
aunt like apple
brother finish all chocolate
brother find minemy chocolate
minemy chocolate where
who finish icecream
who finish milk
who finish pizza
why food not hot
why refrigerator empty
table dirty
table clean
clean table after food
napkin table
napkin dirty
give minemy napkin please
face dirty napkin
drink water
drink milk
drop milk
milk drop
child drop icecream cry
child drop apple
child drop pizza
water hot
water not hot
thirsty drink water now
hot water bath
store have apple
store have milk
store have icecream
go store milk apple
dad go store food
mom go store find chocolate
weus go store later food
tomorrow weus make pizza
yesterday weus have pizza
now weus have icecream
later weus have snack
chocolate before food no
no chocolate before food
no icecream now
no gum
yes please
no thankyou
thankyou mom
thankyou dad food
thankyou grandma chocolate
taste yucky
taste fine
taste same
'''
_CORPUS_FILES['sentences.v1/home.txt'] = r'''# Home: rooms, furniture, chores, bedtime and bathroom routine.
minemy room dirty haveto clean now
minemy room clean
minemy bedroom up stairs
minemy bedroom quiet
minemy bed wet
minemy bed hot
bedroom dirty mom mad
bedroom lamp open
lamp not on
lamp on table
lamp beside bed
TV on
TV loud
TV noisy close please
TV close now
TV after food
no TV before bath
minemy like TV
dad look TV night
grandpa look TV chair
weus look TV after snack
child look TV morning
brother look TV all night
yourself look TV many time
radio loud
radio quiet please
radio on table
grandma listen radio morning
dad hear radio car
child hear radio dance
glasswindow open
glasswindow close
glasswindow dirty
glasswindow dirty haveto clean
close glasswindow rain
open glasswindow hot
look outside glasswindow
child look glasswindow snow
cat look outside glasswindow
bird beside glasswindow
drawer open
drawer close
drawer empty
drawer stuck
drawer stuck not open
minemy shirt drawer
scissors drawer
pencil pen drawer
find scissors drawer
where scissors
scissors where
who have scissors
scissors cut hair
closet open
closet close
closet empty
minemy jacket closet
shoe closet dirty
hat mitten closet
brother hide closet quiet
child hide closet quiet
where hide closet
dryer loud
dryer hot
dryer finish
shirt jeans dryer
pajamas dryer
shirt into dryer
jeans wet into dryer
dryer shirt dry
shirt wet not dry
garbage many
garbage empty
garbage yucky
garbage outside
garbage dirty yucky
dad go garbage outside
brother haveto go garbage outside
drop garbage into garbage
vacuum loud
vacuum noisy
mom vacuum room
dad vacuum stairs
dog hate vacuum
cat hide because vacuum loud
vacuum finish now
stairs high
stairs up
stairs down
go up stairs
go down stairs
child fall stairs owie
child fall down stairs cry
grandma not fast stairs
child wait stairs
table chair
chair beside table
chair dirty
grandpa sleep chair night
grandpa chair old
child jump chair bad
no jump bed
no jump chair
child jump bed fall
bed clean
minemy bed old
time bed now
bed time now
time sleep now
haveto sleep now
child not sleepy
child awake night
child wake night cry
child sleepy haveto nap
nap time now
nap finish
child wake after nap happy
pajamas on bed
pajamas where
pajamas on now
bath time
bath time now
bath first after bed
bath hot
bath water hot
bath water not hot
bath finish
child like bath
child hate bath cry
dog hate bath
shower hot
shower not hot
dad shower morning
mom shower night
shower first after food
potty where please
potty now
haveto potty
child haveto potty now
where potty
potty up stairs
potty finish
toothbrush where
toothbrush blue
toothbrush red
minemy toothbrush green
toothbrush tooth before bed
toothbrush before sleep
tooth owie
child tooth drop
child tooth fall
home now
go home
go home now
weus go home
stay home
stay home sick
stay home now rain
minemy home
minemy home backyard
backyard grass
backyard tree
backyard flower
backyard pool
weus go backyard
dog backyard
kitty backyard tree
child jump backyard
child dance backyard
home quiet
home noisy
home clean
home dirty
home hot
'''
_CORPUS_FILES['sentences.v1/play.txt'] = r'''# Toys, games, books, vehicles, outings.
minemy doll pretty hair yellow
minemy doll where find please
girl give doll another girl
girl pretend doll sleep bed
doll drop girl cry
doll dirty haveto bath
grandma make doll for girl
minemy doll old like
child read book doll
minemy puzzle many animal
puzzle finish happy
weus make puzzle table
puzzle drop all
puzzle another find
dad give puzzle for brother
grandpa like puzzle quiet
puzzle animal zebra giraffe
puzzle car airplane boat
book read please
read book for minemy
mom read book before sleep
grandpa read story for child
read story another please
story finish now sleep
minemy like story lion
story wolf pig
story alligator bad
story cowboy horse
story fireman police
story frog duck pool
yourself read book
yourself like story
who read story
another story please
book there table
book drawer
book red blue
pen blue
pen red
pen drawer
pencil yellow
pencil drawer
pen pencil table
minemy pen where
who have minemy pencil
find pencil for brother
child pencil drop
pencil drop beside table
give pen please
penny find beside chair
child find penny
penny brown
many penny
grandpa give penny for child
penny drop garbage
child have many penny happy
gift for minemy
gift for mom
gift for grandma
gift pretty
gift open now
gift open after food
gift many
gift another
who give gift
uncle give gift for brother
aunt give gift for girl
yesterday grandpa give minemy gift
minemy like gift thankyou
thankyou gift grandma
balloon red
balloon blue
balloon yellow many
balloon high
balloon go up high
balloon drop
balloon drop loud
clown blow balloon red
blow balloon
blow balloon many
dad blow balloon for child
child blow balloon
child cry balloon go up
flag red white blue
flag high
flag outside
flag pretty
police flag
child have flag
flag beside home
airplane high
airplane loud
airplane go fast
airplane cloud
weus ride airplane
uncle ride airplane tomorrow
child look airplane high
boy like airplane helicopter
helicopter loud
helicopter high
helicopter noisy
helicopter go fast
police helicopter
fireman helicopter
look helicopter up
child look helicopter happy
boat water
boat blue
boat go fast water
weus ride boat
grandpa have boat
dad boat fish
tomorrow weus ride boat
car red
car fast
car dirty
car stuck snow
car stuck rain
car noisy
dad car
mom car blue
police car noisy
weus ride car grandma home
child like ride car
ride horse farm
ride boat
ride airplane
ride car
pool water
pool hot
pool wet
weus go pool after food
jump into pool
child jump into pool happy
dog jump into pool
frog jump into pool
yesterday weus go pool happy
backyard pool wet
grass green
grass wet
grass dry
grass high
dog grass
child dance grass
frog grass
bug grass
cow grass
horse grass
dance now please
dance happy
dance loud radio
girl dance pretty
weus dance
grandma dance grandpa
child dance TV
pretend lion loud
pretend police
pretend fireman
pretend cowboy
pretend mom
pretend cat
pretend dog
pretend lion
pretend sleep
toy many closet
toy car
toy airplane
toy boat
toy many
toy dirty
toy drop
toy where
toy closet
toy drawer
clean toy now
minemy toy
who have minemy toy
give toy for another child
toy give another child
'''
_CORPUS_FILES['sentences.v1/stories.txt'] = r'''# Longer multi-clause sentences, little stories, lists and chained events.
# Written last, to bring every gloss up to the corpus's minimum coverage
# (validate(min_per_gloss=12)) while pulling the length distribution upward.

# face and body
child fall down stairs chin owie cry
mom kiss chin owie better now
food on chin napkin please
yourself chin dirty napkin clean now
dad touch chin think
grandpa chin white hair old
chin lips nose face
touch chin touch nose touch ear
child fall down chin arm owie
child chin sticky icecream napkin
chin cheek red hot sick
chin mouth dirty chocolate
ear owie child cry night
dog ear brown cute
minemy ear hear loud radio
grandpa ear not hear
elephant ear high
kitty ear touch smile
listen ear quiet please
lips dry drink water
lips red pretty mom
lips sticky gum yucky
mom kiss cheek lips red
girl lips pretty smile
tongue lips taste icecream
touch lips quiet shhh
head owie sick bed
minemy head hot sick
hat on head snow outside
dog head beside feet sleep
cow head brown white
tongue taste frenchfries hot
dog tongue drink water
tongue red chocolate
child tongue touch icecream
tongue owie hot pizza
arm owie fall bed
arm dirty clean shower
bee touch arm owie cry
cheek red snow outside
grandma kiss cheek bye
nose red clown pretend
dog nose wet
elephant nose water drink
feet wet rain shoe
feet dirty grass backyard
eye close sleep night
eye open wake morning
face napkin clean after food
finger touch icecream sticky
mouth open tooth toothbrush
toothbrush tooth morning night
kiss mom dad bye

# rooms and furniture
lamp beside bed bedroom
lamp on table beside chair
lamp open read book night
lamp close sleep now
dad find lamp closet
minemy lamp yellow pretty
lamp drop owie
cat drop lamp mom mad
lamp high table
bedroom dirty clean vacuum now
minemy bedroom beside brother room
grandma bedroom quiet nap
child hide bedroom closet
bedroom glasswindow open hot
room dirty mom mad vacuum
room clean happy mom
room hot open glasswindow
room many toy dirty
vacuum room after food
vacuum loud dog hide bedroom
vacuum stairs up down
vacuum finish room clean
potty time before bed
potty after food first
child potty finish clean
child haveto potty now please
underwear wet potty
underwear clean drawer bedroom
underwear pajamas dryer dry
dryer dry shirt jeans
jeans dry after dryer
hair dry after shower
shower hot dad morning
shower before bed night
garbage empty now after food
garbage outside dad night
garbage yucky dog find
refrigerator empty store later
drawer empty find scissors
cereal empty store haveto go
radio on dance now
radio loud shhh child sleep
puzzle table finish night
scissors cut hair clown
scissors drawer high child not touch
scissors cut napkin
scissors no touch owie
pen pencil scissors drawer
pen blue pencil yellow
pen drop beside chair
pencil drawer beside scissors
napkin table beside food
napkin for face after pizza
napkin dirty garbage
give napkin for grandpa
napkin wet water drop

# animals, farm, outside
hen goose duck farm grandma
hen brown farm grandma
hen loud morning farm
hen cow pig horse farm many animal
hen sleep night farm
hen find bug grass
hen white brown
cat look hen farm
goose loud mad
goose white duck yellow
goose into water
goose find bug
goose bad boy cry
donkey horse same
donkey carrot hungry
donkey ride farm
donkey mad loud
donkey old grandpa farm
donkey brown beside horse
donkey noisy
giraffe high tree
giraffe drink water
giraffe cute yellow brown
giraffe tongue black
giraffe look zebra elephant
tiger lion zebra outside
tiger orange black pretty
tiger hungry mad
tiger sleep sun
tiger cute puppy same
tiger fast go
alligator into water hide
alligator mouth open tooth
alligator green water
alligator bad no touch
alligator hungry fish
elephant old sleep
elephant beside giraffe
elephant blow water
elephant drink many water
pig dirty pretend
pig dirty hungry
pig farm hungry cow
pig snack carrot
pig sleep farm
wolf loud night moon
wolf hungry pig story
wolf look pig
wolf black night
wolf hide tree
owl night moon tree
owl awake night sleep morning
owl eye yellow high tree
owl say who who
moon night pretty high
moon white cloud night
bird yellow high tree
bird beside glasswindow morning
bird go up high
bird blue red
bee flower yellow
bee bug outside
bug on arm bee
bug black flower
bug frog hungry
frog green jump pool
frog into water
frog pretend jump
fish orange into water
fish blue water boat
fish for kitty
duck yellow into pool
duck water cute
cow brown milk farm
cow black white farm
zebra black white horse same
zebra beside giraffe
flower yellow orange pretty
flower red for mom
flower green grass backyard
flower give grandma pretty
orange carrot same orange
orange cut dad
nuts brown mouse hide drawer
nuts snack hungry
nuts drop beside table
nuts for bird tree
nuts mouse drawer
flag red white blue high
flag beside tree outside
flag cowboy horse
flag fireman car
flag high pretty
cowboy hat brown horse ride
cowboy jeans hat brown
cowboy man farm
clown blow balloon yellow child
blow balloon red yellow
blow nose
blow balloon for girl
blow water hot food
lion loud mad tiger
lion hungry animal
helicopter loud high cloud
helicopter police look find
fireman helicopter high
person there wait
person man police
person beside car
person talk callonphone
man find penny beside car
man cut grass
man black hat
green grass green tree
frog green jump grass
fall snow down
fall down stairs
child fall into pool
cereal milk empty morning
cereal yellow
carrot orange yucky
carrot for horse donkey
cut carrot dad
cut hair mom
cut napkin scissors
empty garbage outside now
garbage empty dad
frenchfries hot yucky
frenchfries sticky
frenchfries pizza chocolate icecream
frenchfries give please
sticky gum hair
finger sticky gum
shhh quiet grandpa nap
shhh listen mom
shhh helicopter
shhh sleep now
nap time child
nap finish happy
kiss cheek grandpa
hear radio
hear helicopter
on table book
on bed pajamas
zipper stuck mom close
zipper jacket black
black cat
black car
cloud black rain later
cloud high airplane
penny drop
fireman hat red jacket yellow
tooth owie cry dad look

# top-ups for the last thin glosses
fish go down water hide
duck goose fish water farm
dog go down stairs
kitty jump down bed
balloon go up high drop down
sun go down night
pen pencil down table
bee go flower yellow
pen pencil puzzle table
alligator wolf zebra story
touch head touch feet touch finger
owl nuts tree night
underwear toothbrush pajamas bed
'''
_CORPUS_FILES['sentences.v1/weather_time.txt'] = r'''# Weather, sky, time words. ASL puts the time reference first.
morning sun
morning sun hot
night moon
night moon white
night moon high pretty
moon night owl awake
sun hot outside
sun yellow
sun high
sun hot weus go pool
sun hot icecream sticky
sun cloud
cloud white
cloud black rain
cloud many rain later
cloud many black rain now
cloud high
cloud look same elephant
cloud look same horse
look cloud high tree
child look cloud pretend animal
rain now
rain outside
rain outside stay home
rain now weus stay home look TV
rain yesterday
rain tomorrow
rain later
if rain weus stay home
if rain yourself haveto jacket
if sun weus go pool
if snow weus go outside
if sun hot weus go backyard
rain grass wet
rain tree wet
rain dog wet
rain child wet cry
rain finish sun hot
rain finish sun
rain finish go outside
rain noisy night
rain loud glasswindow
snow now
snow outside
snow white
snow pretty
snow tomorrow
snow yesterday
snow morning
snow night
snow many
snow outside child happy
snow finish
snow wet mitten
snow hat jacket mitten
yesterday snow weus go outside
tomorrow snow weus stay home
morning wake
morning wake up
morning wake first
morning grandma wake first
morning dad wake child
morning cereal milk
morning shower
morning sun awake
night sleep
night time sleep
night moon pretty
night quiet
night noisy
night owl awake
night wolf awake
night child awake cry
night mom read story
night grandpa read book
now go
now finish
now time sleep
now time food
now time bath
now time nap
later go
later weus go store
later weus look TV
later grandma callonphone
later dad go home
wait later
wait later please
wait now
wait please
wait mom
wait dad
wait minemy
wait there
wait there please
yesterday go farm
yesterday weus go pool
yesterday weus look moon
yesterday grandma give minemy gift
yesterday child cry
yesterday rain all morning
yesterday dad make pizza
yesterday brother fall owie
tomorrow go store
tomorrow weus go farm
tomorrow grandpa go home
tomorrow minemy aunt will callonphone
tomorrow uncle will ride airplane
tomorrow weus will go boat
tomorrow snow will
before sleep bath
before sleep toothbrush
before food napkin
before outside jacket
after food bath
after bath pajamas
after bath story
after story sleep
after nap snack
after snack go outside
after rain sun
first bath after sleep
first food after TV
first toothbrush after bed
first finish food after icecream
time go
time go home
time go now
time sleep
time wake
time food
time bath
time finish
time go home now grandma wait
time finish now
will go
will rain
will snow
minemy will go
hesheit will go
weus will go farm
awake now
awake night
awake morning
child awake
dog awake
not awake
wake now
wake up
wake up now
wake up please
wake up time go
wake dad
wake mom please
'''

CORPUS_TMP = Path(tempfile.mkdtemp(prefix="corpus_"))
for rel, text in _CORPUS_FILES.items():
    (CORPUS_TMP / rel).parent.mkdir(parents=True, exist_ok=True)
    (CORPUS_TMP / rel).write_text(text, encoding="utf-8")
corpus.CORPUS_DIR = CORPUS_TMP
print(f"modules: {list(_SOURCES)}")
print(f"corpus:  {len(_CORPUS_FILES)} files, sha256={corpus.corpus_sha256(CONFIG['corpus_version'])[:12]}")

## 1. Label map + corpus validation

GISLR_Stratified does not ship `sign_to_prediction_index_map.json`. It is
derived exactly as signbridge's `gislr_dir()` does (sorted signs → index), so
the labels here match every signbridge model. Then the corpus is validated and
a sample is printed.

In [ ]:
# ============================================================
# Label map (derived like sb.core.paths.gislr_dir) + corpus validation
# ============================================================
N_SAMPLE = 30

vocab = sys.modules["sb.core.vocab"]
signs = set(pd.read_csv(DATA_DIR / "train.csv")["sign"]) | set(pd.read_csv(DATA_DIR / "test.csv")["sign"])
LABEL_MAP = {sign: i for i, sign in enumerate(sorted(signs))}
(OUT_DIR / vocab.LABEL_MAP_FILE).write_text(json.dumps(LABEL_MAP), encoding="utf-8")

src_df = pd.read_csv(DATA_DIR / f"{CONFIG['source_split']}.csv")
sentences = corpus.load_sentences(CONFIG["corpus_version"])
lexicon = corpus.load_lexicon(CONFIG["corpus_version"])
assert set(lexicon) == set(src_df["sign"]) == set(LABEL_MAP), "lexicon / vocabulary / label map disagree"
report = corpus.validate(sentences, src_df["sign"].unique(), **CONFIG["corpus"])
print(f"{report['n_sentences']} sentences, {report['n_themes']} themes, mean length "
      f"{report['mean_len']:.2f}, lengths {report['length_hist']}")
print(f"missing: {report['missing'] or 'none'}   below minimum: {report['thin'] or 'none'}")
rng = np.random.default_rng(CONFIG["seed"])
for i in rng.choice(len(sentences), N_SAMPLE, replace=False):
    print(f"  {sentences[i].id:<22} {' '.join(g.upper() for g in sentences[i].glosses)}")

## 2. Plan: assign every test clip to a sequence

The same planner, config and seed as the local notebook, so the plan is
identical. It is cached in `plan.json` for resumption within a session.

In [ ]:
# ============================================================
# Plan both splits
# ============================================================
if PLAN_PATH.exists() and json.loads(PLAN_PATH.read_text())["config_sha256"] == CONFIG_SHA:
    PLAN = json.loads(PLAN_PATH.read_text())
    print("plan loaded")
else:
    src_df = pd.read_csv(DATA_DIR / f"{CONFIG['source_split']}.csv")
    rng = np.random.default_rng(CONFIG["seed"])
    synth = dict(gap_range=tuple(CONFIG["synth"]["gap_frames"]),
                 rest_range=tuple(CONFIG["synth"]["rest_frames"]))
    bar = tqdm(total=len(src_df), desc="plan sentence split")
    sent_plan, stats = compose.plan_sentences(
        src_df, corpus.load_sentences(CONFIG["corpus_version"]), rng, **synth, **CONFIG["planner"],
        progress=lambda done, total: bar.update(done - bar.n))
    bar.close()
    ctrl_plan = compose.plan_control(src_df, np.array([len(r["glosses"]) for r in sent_plan]), rng, **synth)
    for i, r in enumerate(sent_plan):
        r["seq_id"] = f"s{i:05d}"
    for i, r in enumerate(ctrl_plan):
        r["seq_id"] = f"c{i:05d}"
    PLAN = {"config_sha256": CONFIG_SHA, "stats": stats, "sequences": sent_plan + ctrl_plan}
    write_json(PLAN_PATH, PLAN)

st = PLAN["stats"]
n_sent = sum(r["kind"] == "sentence" for r in PLAN["sequences"])
print(f"sentence: {n_sent} sequences, {st['n_clips']} clips used, reuse {st['reuse_rate']:.1%}, "
      f"orphans {st['n_orphan_clips']}, {st['n_distinct_sentences']} distinct sentences")
print(f"control:  {len(PLAN['sequences']) - n_sent} sequences")

## 3. Materialize the sequences

Manifest-driven: each npz is written atomically, then marked `done`. Each
sequence's jitter RNG is seeded from `(seed, row index)`. After the first 300
sequences the final size is projected, and the build stops if it would exceed
`OUTPUT_LIMIT_GB`.

In [ ]:
# ============================================================
# Build every sequence npz (resumable, size-guarded)
# ============================================================
PROJECT_AFTER = 300

rows = PLAN["sequences"]
src_df = pd.read_csv(DATA_DIR / f"{CONFIG['source_split']}.csv")
relpath_of = dict(zip(src_df["uid"], src_df["npz_relpath"]))
dtype = np.dtype(STORAGE_DTYPE)
manifest = json.loads(MANIFEST_PATH.read_text()) if MANIFEST_PATH.exists() else {}


def build(i_row):
    i, row = i_row
    rel = f"sequences/{row['kind']}/{row['seq_id']}.npz"
    try:
        arrays = compose.materialize(row, DATA_DIR, relpath_of, LABEL_MAP,
                                     np.random.default_rng([CONFIG["seed"], i]),
                                     rest_jitter=CONFIG["synth"]["rest_jitter"])
        compose.write_sequence(OUT_DIR / rel, arrays, dtype=dtype)
        rec = compose.summarize_row(row, arrays, row["seq_id"], rel)
        return row["seq_id"], {"status": "done", "bytes": (OUT_DIR / rel).stat().st_size, **rec}
    except Exception as e:
        return row["seq_id"], {"status": "failed", "error": repr(e)}


todo = [(i, r) for i, r in enumerate(rows) if manifest.get(r["seq_id"], {}).get("status") != "done"]
bar = tqdm(total=len(rows), initial=len(rows) - len(todo), desc="materialize")
n_failed = 0
with ThreadPoolExecutor(N_WORKERS) as ex:
    for k, (seq_id, rec) in enumerate(ex.map(build, todo), 1):
        manifest[seq_id] = rec
        n_failed += rec["status"] == "failed"
        bar.update(1)
        done_bytes = [v["bytes"] for v in manifest.values() if v["status"] == "done"]
        projected = np.mean(done_bytes) * len(rows) / 1e9 if done_bytes else 0.0
        bar.set_postfix(failed=n_failed, projected_gb=f"{projected:.1f}")
        if k == PROJECT_AFTER and projected > OUTPUT_LIMIT_GB:
            write_json(MANIFEST_PATH, manifest)
            raise RuntimeError(
                f"projected {projected:.1f} GB > OUTPUT_LIMIT_GB={OUTPUT_LIMIT_GB}. Set "
                f"STORAGE_DTYPE = 'float16', delete {SEQ_DIR} and {MANIFEST_PATH}, and re-run.")
        if k % 500 == 0:
            write_json(MANIFEST_PATH, manifest)
write_json(MANIFEST_PATH, manifest)
bar.close()
print(f"{sum(v['status'] == 'done' for v in manifest.values())}/{len(rows)} done, {n_failed} failed this run")

## 4. Index, corpus copy, build record, dataset card

In [ ]:
# ============================================================
# sequences.csv, sentences.csv, lexicon, build_info, README
# ============================================================
manifest = json.loads(MANIFEST_PATH.read_text())
failed = [k for k, v in manifest.items() if v["status"] != "done"]
assert not failed, f"{len(failed)} sequences not done -- re-run section 3: {failed[:5]}"

order = [r["seq_id"] for r in PLAN["sequences"]]
seq_df = pd.DataFrame([{k: v for k, v in manifest[s].items() if k not in ("status", "bytes")} for s in order])
seq_df.to_csv(OUT_DIR / "sequences.csv", index=False)
sentences = corpus.load_sentences(CONFIG["corpus_version"])
pd.DataFrame([{"sentence_id": s.id, "theme": s.theme, "glosses": " ".join(s.glosses)}
              for s in sentences]).to_csv(OUT_DIR / "sentences.csv", index=False)
shutil.copy(corpus.lexicon_path(CONFIG["corpus_version"]), OUT_DIR / corpus.lexicon_path(CONFIG["corpus_version"]).name)

frames = seq_df.groupby("kind")["n_frames"].agg(["count", "sum", "mean", "max"])
size_bytes = sum(p.stat().st_size for p in SEQ_DIR.rglob("*.npz"))
build_info = {
    "dataset": "GISLR-Sentences", "dataset_version": CONFIG["dataset_version"],
    "built_on": "kaggle", "storage_dtype": STORAGE_DTYPE,
    "config": CONFIG, "config_sha256": CONFIG_SHA,
    "corpus_sha256": corpus.corpus_sha256(CONFIG["corpus_version"]),
    "code_sha256": CODE_SHA256, "signbridge_commit": SIGNBRIDGE_COMMIT,
    "source": {"kaggle_ref": "bracu23101281/gislr-stratified", "resolved_dir": str(DATA_DIR),
               "split": CONFIG["source_split"]},
    "planner_stats": PLAN["stats"],
    "frames": frames.reset_index().to_dict(orient="records"),
    "size_bytes": size_bytes,
}
write_json(OUT_DIR / "build_info.json", build_info)

s = PLAN["stats"]
g, r = CONFIG["synth"]["gap_frames"], CONFIG["synth"]["rest_frames"]
card = f"""# GISLR-Sentences {CONFIG['dataset_version']}

Continuous multi-sign landmark sequences built from **real** isolated-sign clips of
GISLR_Stratified's `{CONFIG['source_split']}.csv` (`bracu23101281/gislr-stratified`, 250 glosses). Signed
clips are unmodified; only the non-sign frames between and around them are synthesized.
For evaluating streaming/continuous recognizers and sign-boundary reset (signbridge TODO §12).

## Splits
- **sentence** ({(seq_df.kind == 'sentence').sum()} sequences): {s['n_distinct_sentences']} distinct ASL-gloss-order
  sentences from a {len(sentences)}-sentence corpus. Every source clip is used;
  {s['reuse_rate']:.1%} of sign slots re-use an already-placed clip (flagged in `reused`).
- **control** ({(seq_df.kind == 'control').sum()} sequences): the same clips in random order, each exactly once.

One signer per sequence, always.

## Files
- `sequences/<kind>/<seq_id>.npz`: `landmarks (T,543,3) {STORAGE_DTYPE}` (NaN = not detected; canonical
  GISLR holistic row order: face 0-467, left_hand 468-488, pose 489-521, right_hand 522-542),
  `frame_labels (T,) int16` (-1 = null), `frame_kind (T,) uint8` (0 sign, 1 transition, 2 rest),
  `segments (n,2) int32` (start, end-exclusive), `labels (n,) int16`.
- `sequences.csv`: one row per sequence, with source clip uids and all segment boundaries.
- `sentences.csv`, `lexicon.{CONFIG['corpus_version']}.json`, `sign_to_prediction_index_map.json`, `plan.json`, `build_info.json`.

## Synthesized frames (labelled -1)
- **transition**: {g[0]}-{g[1]} frames between signs, linearly interpolated from the last frame of
  one clip to the first frame of the next.
- **rest**: {r[0]}-{r[1]} frames at each end, with the body held still plus Gaussian jitter
  (sd {CONFIG['synth']['rest_jitter']}) and both hands NaN.

`landmarks[frame_kind == 0]` is the back-to-back concatenation of the source clips.

## Caveats
- The non-sign frames are synthetic; real transitions and rest poses are not in GISLR. Treat
  scores as an upper bound on real continuous signing.
- GISLR_Stratified's test split is also the validation split of the signbridge isolated-sign
  models, so those models were checkpoint-selected on these clips.
- The vocabulary is toddler-level and has no sign for I/me (`minemy` = my/mine).
"""
(OUT_DIR / "README.md").write_text(card, encoding="utf-8")
print(frames)
print(f"size: {size_bytes / 1e9:.2f} GB ({STORAGE_DTYPE})")

## 5. Checks and figures

In [ ]:
# ============================================================
# Round-trip sample + index-level consistency
# ============================================================
by_id = {r["seq_id"]: r for r in PLAN["sequences"]}
seq_df = pd.read_csv(OUT_DIR / "sequences.csv", keep_default_na=False).set_index("seq_id", drop=False)
src_df = pd.read_csv(DATA_DIR / f"{CONFIG['source_split']}.csv")
relpath_of = dict(zip(src_df["uid"], src_df["npz_relpath"]))

rng = np.random.default_rng(CONFIG["seed"])
sample = sorted(rng.choice(seq_df["seq_id"].to_numpy(), CONFIG["n_roundtrip_checks"], replace=False).tolist())
bad = []
for seq_id in tqdm(sample, desc="round-trip"):
    arrays = compose.read_sequence(OUT_DIR / seq_df.loc[seq_id, "npz_relpath"])
    try:
        compose.check_roundtrip(arrays, by_id[seq_id], DATA_DIR, relpath_of)
    except AssertionError as e:
        bad.append((seq_id, str(e)))
write_json(OUT_DIR / "roundtrip_sample.json", {"seed": CONFIG["seed"], "seq_ids": sample, "failures": bad})
print(f"round-trip: {len(sample) - len(bad)}/{len(sample)} exact at {STORAGE_DTYPE}")

issues = 0
for r in seq_df.itertuples():
    s = np.array(r.starts.split(), int)
    e = np.array(r.ends.split(), int)
    issues += not (len(s) == r.n_signs and (e > s).all() and (s[1:] >= e[:-1]).all()
                   and s[0] >= r.rest_in and e[-1] <= r.n_frames - r.rest_out)
print(f"segment-consistency issues across {len(seq_df)} sequences: {issues}")
used = pd.Series(" ".join(seq_df.loc[seq_df.kind == "sentence", "source_uids"]).split()).value_counts()
print(f"sentence split: {used.size}/{len(src_df)} source clips used; uses per clip "
      f"{used.value_counts().sort_index().to_dict()}")
assert not bad and not issues, "checks failed -- do not publish this output"

In [ ]:
# ============================================================
# Figures: lengths + one exemplar sequence
# ============================================================
seq_df = pd.read_csv(OUT_DIR / "sequences.csv", keep_default_na=False)
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8))
for kind, ax in zip(("sentence", "control"), axes[:2]):
    sub = seq_df[seq_df.kind == kind]
    ax.hist(sub["n_frames"], bins=60, color="#4c72b0")
    ax.set_title(f"{kind}: {len(sub)} sequences, mean {sub.n_frames.mean():.0f} frames")
    ax.set_xlabel("frames")
seq_df.groupby("kind")["n_signs"].value_counts().unstack(0).fillna(0).plot.bar(ax=axes[2])
axes[2].set_title("signs per sequence")
fig.tight_layout()
fig.savefig(ASSETS_DIR / "lengths.png", dpi=110)
plt.show()

ex = seq_df[(seq_df.kind == "sentence") & (seq_df.n_signs == 4)].iloc[0]
arr = compose.read_sequence(OUT_DIR / ex["npz_relpath"])
fig, ax = plt.subplots(figsize=(14, 3.5))
for idx, name in ((489 + 15, "left wrist"), (489 + 16, "right wrist")):
    ax.plot(-arr["landmarks"][:, idx, 1].astype(np.float32), label=f"{name} (-y)")
colors = {compose.TRANSITION: "#dddddd", compose.REST: "#f4cccc"}
for t, k in enumerate(arr["frame_kind"]):
    if k in colors:
        ax.axvspan(t - 0.5, t + 0.5, color=colors[k], lw=0)
for (s, e), gl in zip(arr["segments"], ex["glosses"].split()):
    ax.text((s + e) / 2, ax.get_ylim()[1], gl.upper(), ha="center", va="bottom")
ax.set_title(f"{ex['seq_id']} (signer {ex['participant_id']}): grey = transition, pink = rest (hands NaN)")
ax.legend(loc="lower right")
fig.tight_layout()
fig.savefig(ASSETS_DIR / "exemplar.png", dpi=110)
plt.show()

## 6. Clean up, then publish

The build scratch (`manifest.json`, temp files) is removed so the output
folder is exactly the dataset. Then, from the committed version's **Output**
tab: **New Dataset** → title `GISLR-Sentences`. It is private by default.

In [ ]:
# ============================================================
# Remove build scratch from the output folder
# ============================================================
MANIFEST_PATH.unlink(missing_ok=True)
for p in OUT_DIR.rglob("*.tmp*"):
    p.unlink()
n_files = sum(1 for p in OUT_DIR.rglob("*") if p.is_file())
total = sum(p.stat().st_size for p in OUT_DIR.rglob("*") if p.is_file())
print(f"{OUT_DIR}: {n_files} files, {total / 1e9:.2f} GB -- ready to publish as a dataset")